In [95]:
from langchain_mistralai import ChatMistralAI
from langgraph.graph import StateGraph, END, START
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
load_dotenv()

# tools:
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

import requests
import os

In [96]:
llm = ChatMistralAI(model="mistral-small-2506")

In [97]:
search_tool = DuckDuckGoSearchRun()

@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Performs basic arithmetic operations on two numbers
    supported operations are: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by 0 is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}
        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
    except Exception as e:
        return {"error": str(e)}
    

def get_stock_price(symbol: str) -> dict:
    """
    Fetches latest stock price for the given symbol (eg: 'AAPL', 'TSLA')
    using Alpha Vintage with API key in the URL
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=8KVWO5LHF5VOAAA1"
    response = requests.get(url)
    return response.json()


In [98]:
tools = [search_tool,calculator,get_stock_price]
llm_with_tools = llm.bind_tools(tools)

In [99]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [100]:
def chat_node(state: ChatState):
    """ LLM node that may answer or request a tool call """
    message = state["messages"]
    response = llm_with_tools.invoke(message)
    return {"messages": [response]}

tool_node = ToolNode(tools)

In [101]:
graph = StateGraph(ChatState)

graph.add_node("chat_node",chat_node)
graph.add_node("tools",tool_node)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)

chatbot = graph.compile()

response = chatbot.invoke({
    "messages": [HumanMessage(content="what is stock price of AAPL and how much will it cost to but 50AAPL shares")]
})
print(response["messages"][-1].content)


{"Global Quote": {"01. symbol": "AAPL", "02. open": "309.5600", "03. high": "311.8200", "04. low": "307.6700", "05. price": "308.3300", "06. volume": "48000493", "07. latest trading day": "2026-05-26", "08. previous close": "308.8200", "09. change": "-0.4900", "10. change percent": "-0.1587%"}}
